# Spark Streaming Notebook

This notebook is designed for the PostgreSQL + Debezium + Kafka stack in this repository. It shows how to consume change events from Kafka directly in a notebook and then process them with PySpark Structured Streaming.

The sample assumes:
- Docker Compose is running the PostgreSQL, Kafka, Debezium Connect, and Flink services.
- The Debezium PostgreSQL connector has been registered.
- The event generator is producing updates in the database.


## Configure Kafka and Debezium Source

The Debezium connector in this repository uses the topic prefix `dbserver1`. For the `book_references` table, the emitted topic is normally:

```text
dbserver1.public.book_references
```

You can verify the connector and topic from the REST API or with Kafka tooling.

In [ ]:
import os

bootstrap_servers = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka-broker:29092")

print(f"Bootstrap servers: {bootstrap_servers}")

## Stream Process Events with Spark Structured Streaming

This section shows how to read the same topic from PySpark Structured Streaming, parse the Debezium JSON payload, and transform it. In practice, this is a good next step for learning streaming ETL and windowed aggregations.

In [ ]:
import time
from pyspark.sql import SparkSession

def make_batch_printer(topic: str):
    def print_batch(batch_df, batch_id):
        count = batch_df.count()
        print(f"[{topic}] batch {batch_id}: {count} rows")
        if count > 0:
            batch_df.show(truncate=False)
    return print_batch

def kafka_topic_to_console(spark: SparkSession, topic: str):
    df = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "kafka-broker:29092") \
        .option("subscribe", topic) \
        .option("startingOffsets", "earliest") \
        .load()
    query = df.selectExpr("CAST(value AS STRING) as json") \
        .writeStream \
        .outputMode("append") \
        .format("console") \
        .option("truncate", False) \
        .queryName(topic) \
        .start()
    print(f"PySpark Kafka ({topic}) stream started successfully")
    return query

def monitor(queries, interval=5):
    while True:
        for q in queries:
            if not q.isActive:
                exc = q.exception()
                print(f"[{q.name}] STOPPED — exception: {exc}")
                continue
            lp = q.lastProgress
            if lp:
                print(f"[{q.name}] batch={lp['batchId']} "
                      f"inputRows={lp['sources'][0]['numInputRows']} "
                      f"inputRate={lp['sources'][0].get('inputRowsPerSecond', 0):.2f}/s "
                      f"duration={lp['durationMs'].get('triggerExecution', 0)}ms")
            else:
                print(f"[{q.name}] no progress yet (waiting for data)")
        time.sleep(interval)

In [ ]:
spark = SparkSession.builder \
    .appName("debezium-stream") \
    .remote("sc://spark:15002") \
    .getOrCreate()

try:
    references_query = kafka_topic_to_console(spark, "dbserver1.public.book_references")
    inventories_query = kafka_topic_to_console(spark, "dbserver1.public.book_inventories")
    rentals_query = kafka_topic_to_console(spark, "dbserver1.public.book_rentals")
    monitor([references_query, inventories_query, rentals_query])
finally:
    spark.stop()